In [3]:
# MythBench v1.0 — FINAL Inference Script
# Kaggle T4 x2 | ONE MODEL PER SESSION
# DO NOT CREATE NEW VERSIONS OF THIS FILE — edit in place

# ─────────────────────────────────────────────
# CELL 1: Install
# ─────────────────────────────────────────────
# !pip install -q transformers accelerate

# ─────────────────────────────────────────────
# CELL 2: CONFIG — only change MODEL_NAME
# ─────────────────────────────────────────────
import json, re, csv, torch, gc
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

# ── SET ONE OF THESE BEFORE RUNNING ──────────
# MODEL_NAME = "Gemma2B"
# MODEL_NAME = "Qwen18B"
# MODEL_NAME = "SmolLM"
MODEL_NAME = "TinyLlama"
# ─────────────────────────────────────────────

MODEL_PATHS = {
    "Gemma2B":  "google/gemma-2b-it",
    "Qwen18B":  "Qwen/Qwen1.5-1.8B-Chat",
    "SmolLM":   "HuggingFaceTB/SmolLM-1.7B-Instruct",
    "TinyLlama":"TinyLlama/TinyLlama-1.1B-Chat-v1.0",
}

BENCHMARK_PATH = "/kaggle/input/datasets/kevinsam77/mythbench-dataset/MythBench_v10.json"
OUTPUT_PATH    = f"/kaggle/working/mythbench_results_{MODEL_NAME}_v10.csv"

model_path = MODEL_PATHS[MODEL_NAME]
print(f"Model:  {MODEL_NAME} ({model_path})")
print(f"Output: {OUTPUT_PATH}")

# ─────────────────────────────────────────────
# CELL 3: Load benchmark
# ─────────────────────────────────────────────
with open(BENCHMARK_PATH) as f:
    bench = json.load(f)

all_items = (
    [(item, "misconception") for item in bench["misconceptions"]] +
    [(item, "control")       for item in bench["controls"]]
)
print(f"Benchmark {bench['version']}: {len(all_items)} items "
      f"({len(bench['misconceptions'])} misconception + {len(bench['controls'])} control)")

# ─────────────────────────────────────────────
# CELL 4: Prompt builder
# ─────────────────────────────────────────────
def build_prompt(item, model_name):
    opts = "\n".join(f"{k}) {v}" for k, v in item["options"].items())
    if model_name == "Qwen18B":
        return (
            f"<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n"
            f"<|im_start|>user\nAnswer the multiple choice question. "
            f"Reply with ONLY the letter A, B, C, or D.\n\n"
            f"Question: {item['question']}\n{opts}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
    elif model_name == "TinyLlama":
        return (
            f"<|system|>You are a helpful assistant.</s>\n"
            f"<|user|>Answer the multiple choice question. "
            f"Reply with ONLY the letter A, B, C, or D.\n\n"
            f"Question: {item['question']}\n{opts}</s>\n"
            f"<|assistant|>\n"
        )
    elif model_name == "SmolLM":
        return (
            f"<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n"
            f"<|im_start|>user\nAnswer the multiple choice question. "
            f"Reply with ONLY the letter A, B, C, or D.\n\n"
            f"Question: {item['question']}\n{opts}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
    else:  # Gemma2B
        return (
            f"Answer the following multiple choice question. "
            f"Reply with ONLY the letter of the correct answer (A, B, C, or D).\n\n"
            f"Question: {item['question']}\n\n{opts}\n\nAnswer:"
        )

def extract_answer(text):
    text = text.strip().upper()
    m = re.search(r'\b([ABCD])\b', text)
    if m:
        return m.group(1)
    return text[0] if text and text[0] in "ABCD" else "NONE"

# ─────────────────────────────────────────────
# CELL 5: Load model
# ─────────────────────────────────────────────
print(f"\nLoading {MODEL_NAME}...")

config = AutoConfig.from_pretrained(model_path, token=HF_TOKEN, trust_remote_code=True)
if not hasattr(config, 'pad_token_id') or config.pad_token_id is None:
    config.pad_token_id = config.eos_token_id
    print(f"  Patched pad_token_id = {config.eos_token_id}")

tokenizer = AutoTokenizer.from_pretrained(model_path, token=HF_TOKEN, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    model_path, config=config, token=HF_TOKEN,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model.eval()
print("Model loaded.\n")

# ─────────────────────────────────────────────
# CELL 6: Run inference
# ─────────────────────────────────────────────
results = []
for item, item_type in all_items:
    prompt  = build_prompt(item, MODEL_NAME)
    inputs  = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=5, do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
    raw     = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    answer  = extract_answer(raw)
    correct = item["correct"]
    miscon  = item.get("misconception", "N/A")

    results.append({
        "model":                MODEL_NAME,
        "item_id":              item["id"],
        "item_type":            item_type,
        "category":             item.get("category", "N/A"),
        "misconception_label":  item.get("misconception_label", "N/A"),
        "correct_option":       correct,
        "misconception_option": miscon,
        "model_answer":         answer,
        "raw_output":           raw.strip(),
        "is_correct":           answer == correct,
        "is_misconception":     answer == miscon if miscon != "N/A" else False,
    })
    tag = "✓" if answer==correct else ("✗ MISCON" if answer==miscon else f"✗ other({answer})")
    print(f"  {item['id']:<8} [{item_type[:4]}] {tag} | ans={answer} correct={correct}")

# ─────────────────────────────────────────────
# CELL 7: Save CSV
# ─────────────────────────────────────────────
with open(OUTPUT_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
    writer.writeheader()
    writer.writerows(results)
print(f"\nSaved {len(results)} rows → {OUTPUT_PATH}")

# ─────────────────────────────────────────────
# CELL 8: Summary
# ─────────────────────────────────────────────
misc = [r for r in results if r["item_type"]=="misconception"]
ctrl = [r for r in results if r["item_type"]=="control"]

print(f"\n{'='*50}")
print(f"SUMMARY: {MODEL_NAME} on MythBench v1.0")
print(f"{'='*50}")
print(f"\nMisconceptions (n={len(misc)}):")
print(f"  Attraction rate : {sum(1 for r in misc if r['is_misconception']==True)}/{len(misc)}")
print(f"  Correct         : {sum(1 for r in misc if r['is_correct']==True)}/{len(misc)}")
print(f"  Other wrong     : {sum(1 for r in misc if r['is_correct']==False and r['is_misconception']==False)}/{len(misc)}")

print(f"\nControls (n={len(ctrl)}):")
print(f"  Correct         : {sum(1 for r in ctrl if r['is_correct']==True)}/{len(ctrl)}")
print(f"  Wrong           : {sum(1 for r in ctrl if r['is_correct']==False)}/{len(ctrl)}")

answers = [r["model_answer"] for r in results]
counts  = Counter(answers)
total   = len(answers)
print(f"\nAnswer dist: " + "  ".join(f"{o}={counts.get(o,0)}({counts.get(o,0)/total*100:.0f}%)" for o in "ABCD"))
dom = counts.most_common(1)[0]
print(f"Position bias: {'⚠ YES — ' + dom[0] + ' = ' + str(dom[1]) + '/' + str(total) if dom[1]/total > 0.6 else '✓ NO'}")


Model:  TinyLlama (TinyLlama/TinyLlama-1.1B-Chat-v1.0)
Output: /kaggle/working/mythbench_results_TinyLlama_v10.csv
Benchmark 1.0: 96 items (48 misconception + 48 control)

Loading TinyLlama...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

  Patched pad_token_id = 2


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded.

  M-006    [misc] ✓ | ans=B correct=B
  M-008    [misc] ✗ MISCON | ans=B correct=A
  M-010    [misc] ✓ | ans=D correct=D
  M-011    [misc] ✓ | ans=A correct=A
  M-012    [misc] ✗ other(B) | ans=B correct=C
  M-013    [misc] ✗ MISCON | ans=B correct=D
  M-014    [misc] ✗ other(B) | ans=B correct=C
  M-015    [misc] ✗ MISCON | ans=B correct=C
  M-016    [misc] ✓ | ans=B correct=B
  M-003    [misc] ✗ other(D) | ans=D correct=B
  M-B01    [misc] ✗ other(B) | ans=B correct=A
  M-B02    [misc] ✗ MISCON | ans=B correct=C
  M-B05    [misc] ✗ other(B) | ans=B correct=C
  M-B07    [misc] ✗ MISCON | ans=B correct=C
  M-B08    [misc] ✗ other(B) | ans=B correct=D
  M-B13    [misc] ✗ MISCON | ans=B correct=D
  M-B14    [misc] ✓ | ans=B correct=B
  M-B15    [misc] ✗ MISCON | ans=B correct=D
  M-C01    [misc] ✗ other(B) | ans=B correct=D
  M-C03    [misc] ✗ other(B) | ans=B correct=D
  M-C04    [misc] ✓ | ans=B correct=B
  M-C05    [misc] ✗ MISCON | ans=B correct=D
  M-C06    [misc] ✗ M

In [6]:
# MythBench v0.4 — Inference Script v5c (Kaggle T4 x2)
# ONE MODEL PER RUN — set MODEL_NAME below before running

# ─────────────────────────────────────────────
# CELL 1: Install
# ─────────────────────────────────────────────
# !pip install -q transformers accelerate

# ─────────────────────────────────────────────
# CELL 2: Config — SET THIS BEFORE RUNNING
# ─────────────────────────────────────────────
import json, re, csv, torch, gc
from collections import defaultdict, Counter
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

# ── SET ONE OF THESE ──
MODEL_NAME = "Gemma2B"        # change to "Qwen18B" for second run
# MODEL_NAME = "Qwen18B"

MODEL_PATHS = {
    "Gemma2B": "google/gemma-2b-it",
    "Qwen18B":  "Qwen/Qwen1.5-1.8B-Chat",
}

BENCHMARK_PATH = "/kaggle/input/datasets/kevinsam77/mythbench-dataset/MythBench_v10.json"
OUTPUT_PATH = f"/kaggle/working/mythbench_results_{MODEL_NAME}_v10.csv"


model_path = MODEL_PATHS[MODEL_NAME]
print(f"Running: {MODEL_NAME} ({model_path})")
print(f"Output:  {OUTPUT_PATH}")

# ─────────────────────────────────────────────
# CELL 3: Load benchmark
# ─────────────────────────────────────────────
with open(BENCHMARK_PATH) as f:
    bench = json.load(f)

all_items = (
    [(item, "misconception") for item in bench["misconceptions"]] +
    []
)
print(f"Benchmark {bench['version']}: {len(all_items)} items")

# ─────────────────────────────────────────────
# CELL 4: Prompt builder + answer extractor
# ─────────────────────────────────────────────
def build_prompt(item, model_name):
    opts = "\n".join(f"{k}) {v}" for k, v in item["options"].items())
    if model_name == "Qwen18B":
        return (
            f"<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n"
            f"<|im_start|>user\nAnswer the multiple choice question. "
            f"Reply with ONLY the letter A, B, C, or D.\n\n"
            f"Question: {item['question']}\n{opts}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
    else:
        return (
            f"Answer the following multiple choice question. "
            f"Reply with ONLY the letter of the correct answer (A, B, C, or D).\n\n"
            f"Question: {item['question']}\n\n{opts}\n\nAnswer:"
        )

def extract_answer(text):
    text = text.strip().upper()
    m = re.search(r'\b([ABCD])\b', text)
    if m:
        return m.group(1)
    return text[0] if text and text[0] in "ABCD" else "NONE"

# ─────────────────────────────────────────────
# CELL 5: Load model (full GPU for one model)
# ─────────────────────────────────────────────
config = AutoConfig.from_pretrained(model_path, token=HF_TOKEN, trust_remote_code=True)
if not hasattr(config, 'pad_token_id') or config.pad_token_id is None:
    config.pad_token_id = config.eos_token_id
    print(f"Patched pad_token_id = {config.eos_token_id}")

tokenizer = AutoTokenizer.from_pretrained(model_path, token=HF_TOKEN, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    model_path, config=config, token=HF_TOKEN,
    torch_dtype=torch.float16,
    device_map="auto",          # uses both GPUs if needed
    trust_remote_code=True
)
model.eval()
print("Model loaded.")

# ─────────────────────────────────────────────
# CELL 6: Run inference
# ─────────────────────────────────────────────
results = []
for item, item_type in all_items:
    prompt = build_prompt(item, MODEL_NAME)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=5, do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
    raw    = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    answer = extract_answer(raw)
    correct = item["correct"]
    miscon  = item.get("misconception", "N/A")

    results.append({
        "model":                MODEL_NAME,
        "item_id":              item["id"],
        "item_type":            item_type,
        "category":             item.get("category", "N/A"),
        "misconception_label":  item.get("misconception_label", "N/A"),
        "correct_option":       correct,
        "misconception_option": miscon,
        "model_answer":         answer,
        "raw_output":           raw.strip(),
        "is_correct":           answer == correct,
        "is_misconception":     answer == miscon if miscon != "N/A" else False,
    })
    tag = "✓" if answer==correct else ("✗ MISCON" if answer==miscon else f"✗ other({answer})")
    print(f"  {item['id']} | {tag} | ans={answer} correct={correct}")

# ─────────────────────────────────────────────
# CELL 7: Save + quick summary
# ─────────────────────────────────────────────
with open(OUTPUT_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
    writer.writeheader()
    writer.writerows(results)
print(f"\nSaved → {OUTPUT_PATH}")

misc = [r for r in results if r["item_type"]=="misconception"]
ctrl = [r for r in results if r["item_type"]=="control"]
print(f"\n{MODEL_NAME} summary:")
print(f"  Misconceptions: correct={sum(1 for r in misc if r['is_correct']==True)}  "
      f"miscon={sum(1 for r in misc if r['is_misconception']==True)}  "
      f"other={sum(1 for r in misc if r['is_correct']==False and r['is_misconception']==False)}")
print(f"  Controls:       correct={sum(1 for r in ctrl if r['is_correct']==True)}  "
      f"wrong={sum(1 for r in ctrl if r['is_correct']==False)}")

answers = [r["model_answer"] for r in results]
counts  = Counter(answers)
total   = len(answers)
print(f"\n  Answer distribution: " + "  ".join(f"{o}={counts.get(o,0)}({counts.get(o,0)/total*100:.0f}%)" for o in "ABCD"))
if counts.most_common(1)[0][1]/total > 0.6:
    print(f"  ⚠ POSITION BIAS DETECTED")
else:
    print(f"  ✓ No position bias")

Running: Gemma2B (google/gemma-2b-it)
Output:  /kaggle/working/mythbench_results_Gemma2B_v10.csv
Benchmark 1.0: 48 items


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

Model loaded.
  M-006 | ✗ MISCON | ans=C correct=B
  M-008 | ✗ MISCON | ans=B correct=A
  M-010 | ✗ MISCON | ans=B correct=D
  M-011 | ✗ MISCON | ans=B correct=A
  M-012 | ✗ MISCON | ans=A correct=C
  M-013 | ✗ MISCON | ans=B correct=D
  M-014 | ✗ MISCON | ans=D correct=C
  M-015 | ✗ MISCON | ans=B correct=C
  M-016 | ✗ MISCON | ans=A correct=B
  M-003 | ✓ | ans=B correct=B
  M-B01 | ✗ MISCON | ans=C correct=A
  M-B02 | ✗ MISCON | ans=B correct=C
  M-B05 | ✗ MISCON | ans=A correct=C
  M-B07 | ✗ MISCON | ans=B correct=C
  M-B08 | ✗ MISCON | ans=A correct=D
  M-B13 | ✗ MISCON | ans=B correct=D
  M-B14 | ✗ MISCON | ans=D correct=B
  M-B15 | ✗ MISCON | ans=B correct=D
  M-C01 | ✗ MISCON | ans=C correct=D
  M-C03 | ✗ MISCON | ans=A correct=D
  M-C04 | ✗ MISCON | ans=A correct=B
  M-C05 | ✗ MISCON | ans=B correct=D
  M-C06 | ✗ MISCON | ans=A correct=B
  M-C08 | ✗ MISCON | ans=B correct=C
  M-C09 | ✗ MISCON | ans=B correct=C
  M-C10 | ✗ MISCON | ans=C correct=D
  M-C12 | ✗ MISCON | ans=A corr

In [ ]:
# MythBench v0.4 — Inference Script v5 (Kaggle T4 x2)
# Models: Gemma-2B + Phi-2 (with pre-load config fix) + Qwen-1.8B
# Goal: CMO across 3 models on 10 misconceptions + 10 controls

# ─────────────────────────────────────────────
# CELL 1: Install
# ─────────────────────────────────────────────
# !pip install -q transformers accelerate

# ─────────────────────────────────────────────
# CELL 2: Imports + config
# ─────────────────────────────────────────────
import json, re, csv, torch
from collections import defaultdict, Counter
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

MODELS = {
    "Gemma2B": "google/gemma-2b-it",
    "Qwen18B":  "Qwen/Qwen1.5-1.8B-Chat",
    # "Phi2":  "microsoft/phi-2",   # uncomment if Qwen fails
}

BENCHMARK_PATH = "/kaggle/input/datasets/kevinsam77/mythbench-dataset/MythBench_v04.json"
OUTPUT_PATH    = "/kaggle/working/mythbench_results_v05.csv"

# ─────────────────────────────────────────────
# CELL 3: Pre-flight check
# ─────────────────────────────────────────────
from huggingface_hub import model_info
for name, path in MODELS.items():
    try:
        model_info(path, token=HF_TOKEN)
        print(f"  ✓ {name}")
    except Exception as e:
        print(f"  ✗ {name} — {e}")

# ─────────────────────────────────────────────
# CELL 4: Load benchmark
# ─────────────────────────────────────────────
with open(BENCHMARK_PATH) as f:
    bench = json.load(f)

all_items = (
    [(item, "misconception") for item in bench["misconceptions"]] +
    [(item, "control")       for item in bench["controls"]]
)
print(f"Benchmark {bench['version']}: {len(all_items)} items ({len(bench['misconceptions'])} miscon + {len(bench['controls'])} control)")

# ─────────────────────────────────────────────
# CELL 5: Prompt builder + answer extractor
# ─────────────────────────────────────────────
def build_prompt(item, model_name):
    opts = "\n".join(f"{k}) {v}" for k, v in item["options"].items())
    if model_name in ("Phi2",):
        return (
            f"Instruct: Answer the multiple choice question. "
            f"Reply with ONLY the single letter A, B, C, or D.\n\n"
            f"Question: {item['question']}\n{opts}\n\nOutput:"
        )
    elif model_name == "Qwen18B":
        return (
            f"<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n"
            f"<|im_start|>user\nAnswer the multiple choice question below. "
            f"Reply with ONLY the letter A, B, C, or D.\n\n"
            f"Question: {item['question']}\n{opts}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
    else:  # Gemma and others
        return (
            f"Answer the following multiple choice question. "
            f"Reply with ONLY the letter of the correct answer (A, B, C, or D).\n\n"
            f"Question: {item['question']}\n\n{opts}\n\nAnswer:"
        )

def extract_answer(text):
    text = text.strip().upper()
    m = re.search(r'\b([ABCD])\b', text)
    if m:
        return m.group(1)
    return text[0] if text and text[0] in "ABCD" else "NONE"

# ─────────────────────────────────────────────
# CELL 6: Run inference (with pre-load config fix for Phi-2)
# ─────────────────────────────────────────────
def run_model(model_name, model_path, items):
    print(f"\n{'='*50}\nLoading {model_name}\n{'='*50}")

    # Pre-load config and patch pad_token_id BEFORE model init (fixes Phi-2)
    config = AutoConfig.from_pretrained(model_path, token=HF_TOKEN, trust_remote_code=True)
    if not hasattr(config, 'pad_token_id') or config.pad_token_id is None:
        config.pad_token_id = config.eos_token_id
        print(f"  Patched pad_token_id = eos_token_id ({config.eos_token_id})")

    tokenizer = AutoTokenizer.from_pretrained(model_path, token=HF_TOKEN, trust_remote_code=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    model = AutoModelForCausalLM.from_pretrained(
        model_path, config=config, token=HF_TOKEN,
        torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
    )
    model.eval()
    results = []

    for item, item_type in items:
        prompt = build_prompt(item, model_name)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=5, do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )
        raw    = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        answer = extract_answer(raw)
        correct = item["correct"]
        miscon  = item.get("misconception", "N/A")

        results.append({
            "model":                model_name,
            "item_id":              item["id"],
            "item_type":            item_type,
            "category":             item.get("category", "N/A"),
            "misconception_label":  item.get("misconception_label", "N/A"),
            "correct_option":       correct,
            "misconception_option": miscon,
            "model_answer":         answer,
            "raw_output":           raw.strip(),
            "is_correct":           answer == correct,
            "is_misconception":     answer == miscon if miscon != "N/A" else False,
        })
        tag = "✓" if answer==correct else ("✗ MISCON" if answer==miscon else f"✗ other({answer})")
        print(f"  {item['id']} | {tag} | ans={answer} correct={correct}")

    del model; torch.cuda.empty_cache()
    return results

# ─────────────────────────────────────────────
# CELL 7: Run all models + save
# ─────────────────────────────────────────────
all_results = []
for name, path in MODELS.items():
    all_results.extend(run_model(name, path, all_items))

with open(OUTPUT_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(all_results[0].keys()))
    writer.writeheader()
    writer.writerows(all_results)
print(f"\nSaved {len(all_results)} rows → {OUTPUT_PATH}")

# ─────────────────────────────────────────────
# CELL 8: Position bias check
# ─────────────────────────────────────────────
print("\n" + "="*50 + "\nPOSITION BIAS CHECK\n" + "="*50)
biased = []
for name in MODELS:
    answers = [r["model_answer"] for r in all_results if r["model"]==name]
    counts  = Counter(answers)
    total   = len(answers)
    print(f"\n  {name}: ", end="")
    print("  ".join(f"{o}={counts.get(o,0)}({counts.get(o,0)/total*100:.0f}%)" for o in "ABCD"))
    if counts.most_common(1)[0][1]/total > 0.6:
        print(f"  ⚠ BIAS DETECTED")
        biased.append(name)
    else:
        print(f"  ✓ OK")
if biased:
    print(f"\n⚠ Exclude {biased} from CMO — results unreliable")

# ─────────────────────────────────────────────
# CELL 9: CMO + SEA per model pair
# ─────────────────────────────────────────────
by_item = defaultdict(dict)
for r in all_results:
    by_item[r["item_id"]][r["model"]] = r

model_names = [n for n in MODELS if n not in biased]

def compute_cmo_sea(m1, m2, itype):
    bsw = both_w = aow = 0
    for iid, ma in by_item.items():
        if m1 not in ma or m2 not in ma: continue
        if ma[m1]["item_type"] != itype: continue
        a1, a2 = ma[m1]["model_answer"], ma[m2]["model_answer"]
        c = ma[m1]["correct_option"]
        w1, w2 = a1!=c, a2!=c
        if w1 or w2:
            aow += 1
            if w1 and w2:
                both_w += 1
                if a1 == a2: bsw += 1
    cmo = bsw/aow   if aow    else 0.0
    sea = bsw/both_w if both_w else 0.0
    return cmo, sea, bsw, aow, both_w

from itertools import combinations
print("\n" + "="*50 + "\nCMO + SEA RESULTS\n" + "="*50)
for m1, m2 in combinations(model_names, 2):
    print(f"\n  Pair: {m1} vs {m2}")
    print(f"  {'':20} {'Misconception':>15} {'Control':>10} {'Gap':>8}")
    for itype in ["misconception", "control"]:
        cmo, sea, bsw, aow, bw = compute_cmo_sea(m1, m2, itype)
        if itype == "misconception":
            cmo_m, sea_m = cmo, sea
        else:
            cmo_c, sea_c = cmo, sea
    print(f"  {'CMO':<20} {cmo_m:>15.3f} {cmo_c:>10.3f} {cmo_m-cmo_c:>8.3f}")
    print(f"  {'SEA':<20} {sea_m:>15.3f} {sea_c:>10.3f} {sea_m-sea_c:>8.3f}")
    gap = cmo_m - cmo_c
    verdict = "✓ STRONG" if gap>0.20 else ("~ MODERATE" if gap>0.10 else "✗ WEAK")
    print(f"  CMO gap verdict: {verdict}")

# ─────────────────────────────────────────────
# CELL 10: PMD per misconception
# ─────────────────────────────────────────────
print("\n" + "="*50 + "\nPMD (fraction of models choosing misconception)\n" + "="*50)
print(f"{'ID':<8} {'Label':<42} {'PMD':>5}  Models")
print("-"*75)
for item in bench["misconceptions"]:
    iid = item["id"]
    mods = [m for m in model_names if by_item[iid].get(m,{}).get("is_misconception")==True]
    pmd  = len(mods)/len(model_names)
    flag = " ← ALL" if pmd==1.0 else ""
    print(f"{iid:<8} {item['misconception_label']:<42} {pmd:>5.2f}  {mods}{flag}")

In [ ]:
# MythBench v0.3 — Inference Script v4 (Kaggle T4 x2)
# Changes from v3: Phi-2 pad_token_id fix; Gemma-2B only option added

# ─────────────────────────────────────────────
# CELL 1: Install
# ─────────────────────────────────────────────
# !pip install -q transformers accelerate

# ─────────────────────────────────────────────
# CELL 2: Imports + config
# ─────────────────────────────────────────────
import json, re, csv, torch
from collections import defaultdict, Counter
from transformers import AutoTokenizer, AutoModelForCausalLM
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

MODELS = {
    
    "Gemma2B": "google/gemma-2b-it",
}

BENCHMARK_PATH = "/kaggle/input/datasets/kevinsam77/mythbench-dataset/MythBench_v03.json"
OUTPUT_PATH    = "/kaggle/working/mythbench_results_v04.csv"

# ─────────────────────────────────────────────
# CELL 3: Pre-flight access check
# ─────────────────────────────────────────────
from huggingface_hub import model_info
print("Checking model access...")
for name, path in MODELS.items():
    try:
        model_info(path, token=HF_TOKEN)
        print(f"  ✓ {name} ({path})")
    except Exception as e:
        print(f"  ✗ {name} — {e}")

# ─────────────────────────────────────────────
# CELL 4: Load benchmark
# ─────────────────────────────────────────────
with open(BENCHMARK_PATH) as f:
    bench = json.load(f)

all_items = [(item, "misconception") for item in bench["misconceptions"]]
print(f"Benchmark {bench['version']}: {len(all_items)} items")

# ─────────────────────────────────────────────
# CELL 5: Prompt builder + answer extractor
# ─────────────────────────────────────────────
def build_prompt(item, model_name):
    opts = "\n".join(f"{k}) {v}" for k, v in item["options"].items())
    if model_name == "Phi2":
        return (
            f"Instruct: Answer the multiple choice question below. "
            f"Reply with ONLY the single letter A, B, C, or D.\n\n"
            f"Question: {item['question']}\n{opts}\n\nOutput:"
        )
    else:
        return (
            f"Answer the following multiple choice question. "
            f"Reply with ONLY the letter of the correct answer (A, B, C, or D).\n\n"
            f"Question: {item['question']}\n\n{opts}\n\nAnswer:"
        )

def extract_answer(text):
    text = text.strip().upper()
    m = re.search(r'\b([ABCD])\b', text)
    if m:
        return m.group(1)
    return text[0] if text and text[0] in "ABCD" else "NONE"

# ─────────────────────────────────────────────
# CELL 6: Run inference
# ─────────────────────────────────────────────
def run_model(model_name, model_path, items):
    print(f"\n{'='*50}\nLoading {model_name}\n{'='*50}")

    tokenizer = AutoTokenizer.from_pretrained(
        model_path, token=HF_TOKEN, trust_remote_code=True
    )

    # ── FIX: Phi-2 has no pad_token_id in config ──
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
        print(f"  pad_token_id not set — using eos_token_id ({tokenizer.eos_token_id})")

    model = AutoModelForCausalLM.from_pretrained(
        model_path, token=HF_TOKEN,
        torch_dtype=torch.float16, device_map="auto",
        trust_remote_code=True
    )
    # Also patch config directly for Phi-2
    if not hasattr(model.config, 'pad_token_id') or model.config.pad_token_id is None:
        model.config.pad_token_id = tokenizer.eos_token_id

    model.eval()
    results = []

    for item, item_type in items:
        prompt = build_prompt(item, model_name)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=5,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id   # explicit — avoids warning
            )
        raw = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )
        answer  = extract_answer(raw)
        correct = item["correct"]
        miscon  = item.get("misconception", "N/A")

        results.append({
            "model":                model_name,
            "item_id":              item["id"],
            "item_type":            item_type,
            "category":             item.get("category", "N/A"),
            "misconception_label":  item.get("misconception_label", "N/A"),
            "correct_option":       correct,
            "misconception_option": miscon,
            "model_answer":         answer,
            "raw_output":           raw.strip(),
            "is_correct":           answer == correct,
            "is_misconception":     answer == miscon if miscon != "N/A" else False,
        })

        tag = "✓" if answer == correct else ("✗ MISCON" if answer == miscon else f"✗ other({answer})")
        print(f"  {item['id']} | {tag} | model={answer} correct={correct} miscon={miscon}")

    del model; torch.cuda.empty_cache()
    return results

# ─────────────────────────────────────────────
# CELL 7: Run + save
# ─────────────────────────────────────────────
all_results = []
for name, path in MODELS.items():
    all_results.extend(run_model(name, path, all_items))

with open(OUTPUT_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(all_results[0].keys()))
    writer.writeheader()
    writer.writerows(all_results)
print(f"\nSaved {len(all_results)} rows → {OUTPUT_PATH}")

# ─────────────────────────────────────────────
# CELL 8: Position bias check
# ─────────────────────────────────────────────
print("\n" + "="*50)
print("POSITION BIAS CHECK")
print("="*50)
biased = []
for name in MODELS:
    answers = [r["model_answer"] for r in all_results if r["model"] == name]
    counts  = Counter(answers)
    total   = len(answers)
    print(f"\n  {name}:")
    for opt in ["A","B","C","D","NONE"]:
        pct = counts.get(opt, 0) / total * 100
        bar = "█" * int(pct / 5)
        print(f"    {opt}: {counts.get(opt,0):2d} ({pct:4.1f}%) {bar}")
    dominant = counts.most_common(1)[0]
    if dominant[1] / total > 0.6:
        print(f"  ⚠ BIAS: '{dominant[0]}' = {dominant[1]}/{total}")
        biased.append(name)
    else:
        print(f"  ✓ No position bias")

if biased:
    print(f"\n⚠ {biased} biased — CMO below is unreliable for these models")

# ─────────────────────────────────────────────
# CELL 9: Misconception attraction per item
# ─────────────────────────────────────────────
print("\n" + "="*50)
print("MISCONCEPTION ATTRACTION (goal: find items where models choose misconception)")
print("="*50)

by_item = defaultdict(dict)
for r in all_results:
    by_item[r["item_id"]][r["model"]] = r

model_names = list(MODELS.keys())

strong, partial, weak = [], [], []
print(f"\n{'ID':<8} {'Label':<42} {'Attract':>8}  " + "  ".join(f"{m:>10}" for m in model_names))
print("-"*90)
for item in bench["misconceptions"]:
    iid = item["id"]
    attraction = sum(
        1 for m in model_names
        if by_item[iid].get(m, {}).get("is_misconception") == True
    )
    row = f"{iid:<8} {item['misconception_label']:<42} {attraction}/{len(model_names)}  "
    for m in model_names:
        r = by_item[iid].get(m, {})
        ans = r.get("model_answer","?")
        tag = "MISCON" if r.get("is_misconception")==True else ("✓" if r.get("is_correct")==True else f"other({ans})")
        row += f"  {tag:>10}"
    print(row)

    if attraction == len(model_names):
        strong.append(iid)
    elif attraction >= 1:
        partial.append(iid)
    else:
        weak.append(iid)

print(f"\n  Strong (all models → misconception): {len(strong)} — {strong}")
print(f"  Partial (≥1 model → misconception):  {len(partial)} — {partial}")
print(f"  Weak   (no model  → misconception):  {len(weak)} — {weak}")
print(f"\n  Target for proceeding: ≥7 items with attraction ≥1")
hit = len(strong) + len(partial)
print(f"  Current: {hit}/10 items with attraction ≥1  →  {'✓ PROCEED' if hit >= 7 else '✗ REDESIGN MORE ITEMS'}")


In [ ]:
# MythBench v0.2 — Inference Script v3 (Kaggle T4 x2)
# Changes from v2: TinyLlama replaced with Phi-2; added position-bias detector

# ─────────────────────────────────────────────
# CELL 1: Install
# ─────────────────────────────────────────────
# !pip install -q transformers accelerate

# ─────────────────────────────────────────────
# CELL 2: Imports + config
# ─────────────────────────────────────────────
import json, re, csv, torch
from collections import defaultdict, Counter
from transformers import AutoTokenizer, AutoModelForCausalLM
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

MODELS = {
    "Phi2":    "microsoft/phi-2",
    "Gemma2B": "google/gemma-2b-it",
}

BENCHMARK_PATH = "//kaggle/input/datasets/kevinsam77/mythbench-dataset/MythBench_v02.json"
OUTPUT_PATH    = "/kaggle/working/mythbench_results_v03.csv"

# ─────────────────────────────────────────────
# CELL 3: Pre-flight access check
# ─────────────────────────────────────────────
from huggingface_hub import model_info
print("Checking model access...")
for name, path in MODELS.items():
    try:
        model_info(path, token=HF_TOKEN)
        print(f"  ✓ {name} ({path})")
    except Exception as e:
        print(f"  ✗ {name} ({path}) — {e}")
        if "gemma" in path.lower():
            print(f"    → Accept terms at https://huggingface.co/{path}")

# ─────────────────────────────────────────────
# CELL 4: Load benchmark
# ─────────────────────────────────────────────
with open(BENCHMARK_PATH) as f:
    bench = json.load(f)

all_items = (
    [(item, "misconception") for item in bench["misconceptions"]] +
    [(item, "control")       for item in bench["controls"]]
)
print(f"Benchmark {bench['version']}: {len(all_items)} items total")

# ─────────────────────────────────────────────
# CELL 5: Prompt builder + answer extractor
# ─────────────────────────────────────────────
def build_prompt(item, model_name):
    opts = "\n".join(f"{k}) {v}" for k, v in item["options"].items())
    # Phi-2 responds better with Instruct-style format
    if model_name == "Phi2":
        return (
            f"Instruct: Answer the following multiple choice question. "
            f"Reply with ONLY the letter A, B, C, or D.\n\n"
            f"Question: {item['question']}\n{opts}\n\n"
            f"Output:"
        )
    else:
        return (
            f"Answer the following multiple choice question. "
            f"Reply with ONLY the letter of the correct answer (A, B, C, or D).\n\n"
            f"Question: {item['question']}\n\n{opts}\n\nAnswer:"
        )

def extract_answer(text):
    text = text.strip().upper()
    m = re.search(r'\b([ABCD])\b', text)
    if m:
        return m.group(1)
    return text[0] if text and text[0] in "ABCD" else "NONE"

# ─────────────────────────────────────────────
# CELL 6: Run inference for one model
# ─────────────────────────────────────────────
def run_model(model_name, model_path, items):
    print(f"\n{'='*50}\nLoading {model_name}\n{'='*50}")
    tokenizer = AutoTokenizer.from_pretrained(
        model_path, token=HF_TOKEN, trust_remote_code=True
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_path, token=HF_TOKEN,
        torch_dtype=torch.float16, device_map="auto",
        trust_remote_code=True
    )
    model.eval()
    results = []

    for item, item_type in items:
        prompt = build_prompt(item, model_name)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=5, do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        raw = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )
        answer  = extract_answer(raw)
        correct = item["correct"]
        miscon  = item.get("misconception", "N/A")

        results.append({
            "model":                model_name,
            "item_id":              item["id"],
            "item_type":            item_type,
            "category":             item.get("category", "N/A"),
            "misconception_label":  item.get("misconception_label", "N/A"),
            "correct_option":       correct,
            "misconception_option": miscon,
            "model_answer":         answer,
            "raw_output":           raw.strip(),
            "is_correct":           answer == correct,
            "is_misconception":     answer == miscon if miscon != "N/A" else False,
        })

        tag = "✓" if answer == correct else ("✗ MISCON" if answer == miscon else f"✗ other({answer})")
        print(f"  {item['id']} | {tag} | model={answer} correct={correct}")

    del model; torch.cuda.empty_cache()
    return results

# ─────────────────────────────────────────────
# CELL 7: Position bias detector
# ─────────────────────────────────────────────
def check_position_bias(results, model_name):
    answers = [r["model_answer"] for r in results if r["model"] == model_name]
    counts  = Counter(answers)
    total   = len(answers)
    print(f"\n  {model_name} answer distribution (n={total}):")
    for opt in ["A","B","C","D","NONE"]:
        pct = counts.get(opt, 0) / total * 100
        bar = "█" * int(pct / 5)
        print(f"    {opt}: {counts.get(opt,0):3d} ({pct:5.1f}%) {bar}")
    dominant = counts.most_common(1)[0]
    if dominant[1] / total > 0.6:
        print(f"  ⚠ POSITION BIAS DETECTED: '{dominant[0]}' chosen {dominant[1]}/{total} times")
        return True
    print(f"  ✓ No position bias detected")
    return False

# ─────────────────────────────────────────────
# CELL 8: Run all models + save CSV
# ─────────────────────────────────────────────
all_results = []
for name, path in MODELS.items():
    all_results.extend(run_model(name, path, all_items))

with open(OUTPUT_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(all_results[0].keys()))
    writer.writeheader()
    writer.writerows(all_results)
print(f"\nSaved {len(all_results)} rows → {OUTPUT_PATH}")

# ─────────────────────────────────────────────
# CELL 9: Position bias check (gate before CMO)
# ─────────────────────────────────────────────
print("\n" + "="*50)
print("POSITION BIAS CHECK")
print("="*50)
biased_models = []
for name in MODELS:
    biased = check_position_bias(all_results, name)
    if biased:
        biased_models.append(name)

if biased_models:
    print(f"\n⚠ WARNING: {biased_models} show position bias.")
    print("  CMO results below are unreliable for these models.")
    print("  Do not expand benchmark until bias is resolved.")
else:
    print("\n✓ Both models passed bias check. CMO results are reliable.")

# ─────────────────────────────────────────────
# CELL 10: CMO + SEA
# ─────────────────────────────────────────────
answer_map = defaultdict(dict)
for row in all_results:
    answer_map[row["item_id"]][row["model"]] = row

model_names = list(MODELS.keys())

def compute_cmo_sea(m1, m2, item_type_filter):
    both_same_wrong    = 0
    both_wrong         = 0
    at_least_one_wrong = 0
    for iid, ma in answer_map.items():
        if m1 not in ma or m2 not in ma:
            continue
        if ma[m1]["item_type"] != item_type_filter:
            continue
        ans1, ans2 = ma[m1]["model_answer"], ma[m2]["model_answer"]
        correct    = ma[m1]["correct_option"]
        w1, w2     = ans1 != correct, ans2 != correct
        if w1 or w2:
            at_least_one_wrong += 1
            if w1 and w2:
                both_wrong += 1
                if ans1 == ans2:
                    both_same_wrong += 1
    cmo = both_same_wrong / at_least_one_wrong if at_least_one_wrong else 0.0
    sea = both_same_wrong / both_wrong         if both_wrong         else 0.0
    return cmo, sea, both_same_wrong, at_least_one_wrong, both_wrong

print("\n" + "="*50)
print("CMO + SEA RESULTS")
print("="*50)
cmo_m, sea_m, bsw_m, aow_m, bw_m = compute_cmo_sea(model_names[0], model_names[1], "misconception")
cmo_c, sea_c, bsw_c, aow_c, bw_c = compute_cmo_sea(model_names[0], model_names[1], "control")

print(f"\n{'':20} {'Misconception':>15} {'Control':>10} {'Gap':>8}")
print(f"  {'CMO':<18} {cmo_m:>15.3f} {cmo_c:>10.3f} {cmo_m-cmo_c:>8.3f}")
print(f"  {'SEA':<18} {sea_m:>15.3f} {sea_c:>10.3f} {sea_m-sea_c:>8.3f}")
print(f"  {'both_same_wrong':<18} {bsw_m:>15d} {bsw_c:>10d}")
print(f"  {'at_least_one_wrong':<18} {aow_m:>15d} {aow_c:>10d}")

gap = cmo_m - cmo_c
if gap > 0.20:
    verdict = "✓ STRONG — proceed to 5 models + 100-item benchmark"
elif gap > 0.10:
    verdict = "~ MODERATE — add 2 more models before deciding"
elif gap > 0.0:
    verdict = "~ WEAK POSITIVE — inspect items manually"
else:
    verdict = "✗ NO SIGNAL — reconsider premise"
print(f"\n  Verdict: {verdict}")

# ─────────────────────────────────────────────
# CELL 11: PMD preview
# ─────────────────────────────────────────────
print("\n" + "="*50)
print("PMD PREVIEW (exploratory — 2 models)")
print("="*50)
print(f"{'ID':<8} {'Label':<40} {'PMD':>5}  Models")
print("-"*72)
for item in bench["misconceptions"]:
    iid = item["id"]
    models_with = [
        m for m in model_names
        if answer_map[iid].get(m, {}).get("is_misconception") == True  # noqa
    ]
    pmd = len(models_with) / len(model_names)
    flag = " ← both" if pmd == 1.0 else ""
    print(f"{iid:<8} {item['misconception_label']:<40} {pmd:>5.2f}  {models_with}{flag}")
